# Pony Diffusion V6 XL - Uncensored Studio (Verified Non-Gated Mirror)
Uses LyliaEngine/Pony_Diffusion_V6_XL (public, non-gated, verified HTTP 200) via hf_hub_download.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio safetensors huggingface_hub

import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from huggingface_hub import hf_hub_download
import gradio as gr
import os

has_cuda = torch.cuda.is_available()
device = "cuda" if has_cuda else "cpu"
dtype = torch.float16 if has_cuda else torch.float32

print("Downloading Pony Diffusion V6 XL (verified public mirror)...")
model_path = hf_hub_download(
    repo_id="LyliaEngine/Pony_Diffusion_V6_XL",
    filename="ponyDiffusionV6XL_v6StartWithThisOne.safetensors"
)

print("Loading Pony Diffusion V6 XL into pipeline...")
pipe_i2i = StableDiffusionXLImg2ImgPipeline.from_single_file(
    model_path,
    torch_dtype=dtype,
    use_safetensors=True
)

if hasattr(pipe_i2i, "safety_checker"):
    pipe_i2i.safety_checker = None
if hasattr(pipe_i2i, "requires_safety_checker"):
    pipe_i2i.requires_safety_checker = False

if has_cuda:
    pipe_i2i.enable_sequential_cpu_offload()
else:
    pipe_i2i.to("cpu")

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    full_prompt = "score_9, score_8_up, score_7_up, " + prompt
    full_neg = "score_6, score_5, score_4, " + negative_prompt
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=full_neg,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image"),
        gr.Textbox(label="Prompt (tag format e.g. solo, expressive, ...)", value="solo, masterpiece, highly detailed"),
        gr.Textbox(label="Negative Prompt", value="source_low, lowres, bad anatomy"),
        gr.Slider(0.1, 0.6, value=0.25, label="Strength (0.2 - 0.3 preserves identity)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.0, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Result (Uncensored)"),
    title="Pony Diffusion V6 XL Uncensored Studio"
)
demo.launch(share=True, debug=True)
